# YZTA Datathon — v9.1 (Bug Fix)

## v9'dan tek fark: CatBoost `cat_features` hatası düzeltildi

**Hata:** `cat_features` parametresi `cat_params` dict'ine konulmuştu.  
CatBoost bu parametreyi dict'ten değil, `Pool()` nesnesinden ya da `.fit()` çağrısından alır.

**Çözüm:** CatBoost için `Pool` nesnesi oluşturularak kategorik sütunlar oraya bildirildi.  
Bu sayede CatBoost kategorikleri native olarak işler — float'a çevirmeye çalışmaz.

In [13]:
# HÜCRE 1 — Kurulum
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'lightgbm', 'catboost', 'xgboost', '-q'],
    capture_output=True
)

import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings('ignore')
SEED    = 42
N_FOLDS = 5
np.random.seed(SEED)
print('Kütüphaneler yüklendi.')

Kütüphaneler yüklendi.


In [14]:
# HÜCRE 2 — Veri Yükleme + NaN Bayrakları
train_raw = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')

test_raw = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')

sample_sub = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/sample_submission.csv')

test_id = test_raw['id'].copy()
target  = train_raw['bilissel_performans_skoru'].copy()

TARGET_P1  = target.quantile(0.01)
TARGET_P99 = target.quantile(0.99)

print(f'Train: {train_raw.shape} | Test: {test_raw.shape}')
print(f'Target — Min:{target.min():.2f}  Max:{target.max():.2f}  '
      f'Ort:{target.mean():.2f}  Std:{target.std():.2f}')
print(f'Clip: [{TARGET_P1:.4f}, {TARGET_P99:.4f}]')

# NaN bayrakları — encoding öncesi ham veriden çıkar
NAN_FLAG_COLS = ['stres_skoru', 'yas', 'vucut_kitle_indeksi',
                 'gunluk_adim_sayisi']

train_nan = pd.DataFrame()
test_nan  = pd.DataFrame()
for col in NAN_FLAG_COLS:
    if col in train_raw.columns:
        train_nan[f'{col}_nan'] = train_raw[col].isna().astype(int)
        test_nan[f'{col}_nan']  = test_raw[col].isna().astype(int)
        n = train_raw[col].isna().sum()
        if n > 0:
            print(f'  {col}: {n} NaN bulundu')

print(f'NaN flag sütunları: {list(train_nan.columns)}')

Train: (56000, 24) | Test: (24000, 23)
Target — Min:0.00  Max:10.00  Ort:5.91  Std:2.23
Clip: [0.3328, 10.0000]
  stres_skoru: 1715 NaN bulundu
  vucut_kitle_indeksi: 1752 NaN bulundu
NaN flag sütunları: ['stres_skoru_nan', 'yas_nan', 'vucut_kitle_indeksi_nan', 'gunluk_adim_sayisi_nan']


In [15]:
# HÜCRE 3 — Temizlik & Encoding
# İKİ ayrı DataFrame tutulur:
#   train / test       → LGB + XGB için (target encoded kategorikler)
#   train_c / test_c   → CatBoost için (ham string kategorikler)

train = train_raw.drop(columns=['id', 'bilissel_performans_skoru']).copy()
test  = test_raw.drop(columns=['id']).copy()

ulke_mapping   = {'spain':'ispanya', 'south korea':'guney kore', 'sweden':'isvec',
                  'netherlands':'hollanda', 'mexico':'meksika', 'china':'cin'}
meslek_mapping = {'lawyer': 'avukat'}

cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()

for col in cat_cols:
    train[col] = train[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()
    test[col]  = test[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()

train['ulke']   = train['ulke'].replace(ulke_mapping)
test['ulke']    = test['ulke'].replace(ulke_mapping)
train['meslek'] = train['meslek'].replace(meslek_mapping)
test['meslek']  = test['meslek'].replace(meslek_mapping)

# CatBoost kopyası — ham kategorikler korunuyor (sayısal fill sonrası)
train_c = train.copy()
test_c  = test.copy()

# Sayısal eksikler → medyan
for col in num_cols:
    med        = train[col].median()
    train[col] = train[col].fillna(med)
    test[col]  = test[col].fillna(med)
    train_c[col] = train_c[col].fillna(med)
    test_c[col]  = test_c[col].fillna(med)

# Aykırı değer clip
''' for col in num_cols:
    lo, hi = train[col].quantile(0.01), train[col].quantile(0.99)
    train[col]   = train[col].clip(lo, hi)
    test[col]    = test[col].clip(lo, hi)
    train_c[col] = train_c[col].clip(lo, hi)
    test_c[col]  = test_c[col].clip(lo, hi) '''

# Binary encoding (ikisi için de)
for col in ['cinsiyet', 'gun_tipi']:
    cats = sorted(pd.concat([train[col], test[col]], ignore_index=True).unique())
    mp   = {c: i for i, c in enumerate(cats)}
    train[col]   = train[col].map(mp)
    test[col]    = test[col].map(mp)
    train_c[col] = train_c[col].map(mp)
    test_c[col]  = test_c[col].map(mp)

# CatBoost için kategorik sütunlar STRING olarak kalsın
CAT_NATIVE_COLS = ['kronotip', 'ruh_sagligi_durumu', 'meslek', 'ulke', 'mevsim']
for col in CAT_NATIVE_COLS:
    train_c[col] = train_c[col].astype(str)
    test_c[col]  = test_c[col].astype(str)

# Target encoding — sadece LGB/XGB için (train ve test)
SMOOTH      = 15
global_mean = target.mean()
kf_enc      = KFold(n_splits=5, shuffle=True, random_state=SEED)

for col in CAT_NATIVE_COLS:
    agg    = pd.DataFrame({'col': train[col].values, 'target': target.values})
    stats  = agg.groupby('col')['target'].agg(['mean', 'count'])
    sm_map = ((stats['count'] * stats['mean'] + SMOOTH * global_mean)
              / (stats['count'] + SMOOTH)).to_dict()

    oof_enc = np.full(len(train), global_mean, dtype=np.float64)
    for tr_i, va_i in kf_enc.split(train):
        fs = agg.iloc[tr_i].groupby('col')['target'].agg(['mean', 'count'])
        fm = ((fs['count'] * fs['mean'] + SMOOTH * global_mean)
              / (fs['count'] + SMOOTH)).to_dict()
        oof_enc[va_i] = train[col].iloc[va_i].map(fm).fillna(global_mean).values

    train[col] = oof_enc
    test[col]  = test[col].map(sm_map).fillna(global_mean)

print('Encoding tamam.')
print(f'LGB/XGB train: {train.shape} | CatBoost train: {train_c.shape}')

Encoding tamam.
LGB/XGB train: (56000, 22) | CatBoost train: (56000, 22)


In [16]:
# HÜCRE 4 — Feature Engineering v9

def add_features(df, global_stats=None, is_train=True):
    d = df.copy()

    # ── Temel etkileşimler (v3-v6) ────────────────────────────────────────────
    d['uyku_kalite_endeksi']    = ((d['rem_yuzdesi'] + d['derin_uyku_yuzdesi'])
                                    / (d['gecelik_uyanma_sayisi'] + 1))
    d['toplam_kaliteli_uyku']   = d['rem_yuzdesi'] + d['derin_uyku_yuzdesi']
    d['uyku_bozulma_skoru']     = d['gecelik_uyanma_sayisi'] * d['uykuya_dalma_suresi_dk']
    d['ekran_kafein']           = d['uyku_oncesi_ekran_suresi_dk'] * d['uyku_oncesi_kafein_mg']
    d['stres_uyku_orani']       = d['stres_skoru'] / (d['uyku_kalite_endeksi'] + 1)
    d['yas_stres']              = d['yas'] * d['stres_skoru']
    d['meslek_gun_tipi'] = d['meslek'].astype(str) + "_" + d['gun_tipi'].astype(str)
    d['stres_aktivite_dengesi'] = d['gunluk_adim_sayisi'] / (d['stres_skoru'] + 1)
    d['bmi_aktivite']           = d['vucut_kitle_indeksi'] / (d['gunluk_adim_sayisi'] / 1000 + 1)
    d['stres_uyku_gecikme']     = d['stres_skoru'] * d['uykuya_dalma_suresi_dk']
    d['kafein_uyanma_birikimi'] = d['uyku_oncesi_kafein_mg'] * (d['gecelik_uyanma_sayisi'] + 1)
    d['derin_uyku_orani']       = d['derin_uyku_yuzdesi'] / (d['rem_yuzdesi'] + 1)
    d['uyku_etkinlik_skoru'] = (d['toplam_kaliteli_uyku'] /
                               (d['uykuya_dalma_suresi_dk'] / 60 +
                                d['toplam_kaliteli_uyku'] + 1))
    d['uyku_ortam_riski'] = abs(d['oda_sicakligi_celsius'] - 20)
    # Train setinden bulduğumuz eşik değer: 6.61
    d['gizli_risk_grubu'] = ((d['ruh_sagligi_durumu'] == 'Saglikli') & 
                         (d['stres_skoru'] > 6.61) & 
                         (d['gunluk_calisma_saati'] > 8.0)).astype(int)
    # ── v7 polinom + log ──────────────────────────────────────────────────────
    d['geceden_kalma_hasar'] = (d['uyku_oncesi_kafein_mg'] +
                                 d['uyku_oncesi_ekran_suresi_dk']) * d['stres_skoru']

    # ── v9: Stres segment + uyku etkinlik ────────────────────────────────────
    d['stres_segment']       = pd.cut(d['stres_skoru'],
                                       bins=[0, 3, 6, 8, 11],
                                       labels=[0, 1, 2, 3]).astype(float).fillna(1.0)
    d['yuksek_stres']        = (d['stres_skoru'] > 7).astype(int)
    d['dusuk_stres']         = (d['stres_skoru'] < 3).astype(int)
    

    if 'hafta_sonu_uyku_farki_saat' in d.columns:
        d['hafta_sonu_stres'] = d['hafta_sonu_uyku_farki_saat'] * d['stres_skoru']

    return d, stats


# LGB/XGB için
train_fe, global_stats = add_features(train, is_train=True)
test_fe,  _            = add_features(test,  global_stats=global_stats, is_train=False)

# CatBoost için (ham kategoriklerle)
train_cfe, _ = add_features(train_c, global_stats=global_stats, is_train=False)
test_cfe,  _ = add_features(test_c,  global_stats=global_stats, is_train=False)

# NaN bayraklarını ekle (reset_index şart — concat hatası önler)
train_fe  = pd.concat([train_fe.reset_index(drop=True),
                        train_nan.reset_index(drop=True)], axis=1)
test_fe   = pd.concat([test_fe.reset_index(drop=True),
                        test_nan.reset_index(drop=True)], axis=1)
train_cfe = pd.concat([train_cfe.reset_index(drop=True),
                        train_nan.reset_index(drop=True)], axis=1)
test_cfe  = pd.concat([test_cfe.reset_index(drop=True),
                        test_nan.reset_index(drop=True)], axis=1)

print(f'Feature engineering tamam.')
print(f'LGB/XGB: {train_fe.shape[1]} özellik | CatBoost: {train_cfe.shape[1]} özellik')

Feature engineering tamam.
LGB/XGB: 46 özellik | CatBoost: 46 özellik


In [17]:
import numpy as np

# Sadece sayısal özellikleri seç
sayisal_df = train_fe.select_dtypes(include=[np.number])

# Mutlak korelasyon matrisini hesapla
corr_matrix = sayisal_df.corr().abs()

# Sadece üst üçgeni al (Kendisiyle korelasyonları (1.0) ve tekrarları elemek için)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# %90'dan fazla korelasyonu olan sütunları bul
korele_ozellikler = [column for column in upper_tri.columns if any(upper_tri[column] > 0.90)]

print("Aşırı Korele (Tehlikeli) Özellikler Listesi:")
print(korele_ozellikler)

Aşırı Korele (Tehlikeli) Özellikler Listesi:
[]


In [18]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# HÜCRE 4.5 — Hata Önleyici Label Encoding (Sadece LGB/XGB seti için)
# train_fe içindeki metin (object) tipli değişkenleri bul
object_cols = train_fe.select_dtypes(include=['object']).columns.tolist()

for col in object_cols:
    le = LabelEncoder()
    # Train ve Test'i birleştirip fit ediyoruz ki aynı kelimeye aynı sayı atansın
    le.fit(pd.concat([train_fe[col], test_fe[col]]).astype(str))
    
    train_fe[col] = le.transform(train_fe[col].astype(str))
    test_fe[col] = le.transform(test_fe[col].astype(str))

print(f"Sayıya çevrilen metin sütunları: {object_cols}")

Sayıya çevrilen metin sütunları: ['meslek_gun_tipi']


In [19]:
# HÜCRE 5 — Feature Selection

selector = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    num_leaves=31, random_state=SEED, verbose=-1
)
selector.fit(train_fe, target)

imp_df = (pd.DataFrame({'ozellik': train_fe.columns,
                         'importance': selector.feature_importances_})
            .sort_values('importance', ascending=False))
imp_df['katkı_%'] = (imp_df['importance'] / imp_df['importance'].sum() * 100).round(2)

print('Feature Importance (ilk 30):')
print(imp_df.head(30).to_string(index=False))

''' # Sıfır importance'lıları at (sadece LGB/XGB'den)
drop_cols = imp_df[imp_df['importance'] == 0]['ozellik'].tolist()
if drop_cols:
    print(f'\nAtılan: {drop_cols}')
    train_fe = train_fe.drop(columns=drop_cols)
    test_fe  = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns])'''

print(f'\nKalan özellik (LGB/XGB): {train_fe.shape[1]}')
print(f'CatBoost özellik sayısı: {train_cfe.shape[1]} (değişmedi)') 

Feature Importance (ilk 30):
                    ozellik  importance  katkı_%
                rem_yuzdesi         688     4.59
         ruh_sagligi_durumu         687     4.58
                stres_skoru         644     4.29
        uyku_etkinlik_skoru         570     3.80
           uyku_ortam_riski         556     3.71
       gunluk_calisma_saati         548     3.65
           stres_uyku_orani         517     3.45
     stres_aktivite_dengesi         493     3.29
                     meslek         492     3.28
        vucut_kitle_indeksi         485     3.23
        sekerleme_suresi_dk         478     3.19
       toplam_kaliteli_uyku         460     3.07
uyku_oncesi_ekran_suresi_dk         427     2.85
                  yas_stres         425     2.83
         derin_uyku_yuzdesi         423     2.82
         stres_uyku_gecikme         416     2.77
           hafta_sonu_stres         398     2.65
      oda_sicakligi_celsius         392     2.61
           derin_uyku_orani         392 

In [20]:
# HÜCRE 6 — StratifiedKFold + Veri Hazırlık

X   = train_fe.copy()
Xte = test_fe.copy()
Xc  = train_cfe.copy()
Xct = test_cfe.copy()
y   = target.copy()

y_bins   = pd.qcut(y, q=10, labels=False, duplicates='drop')
kf_model = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f'StratifiedKFold: {N_FOLDS} fold, 10 target bin')
print('Fold dağılımı:')
for i, (tr_i, va_i) in enumerate(kf_model.split(X, y_bins), 1):
    print(f'  Fold {i}: val={len(va_i)}, val_target_ort={y.iloc[va_i].mean():.3f}')

StratifiedKFold: 5 fold, 10 target bin
Fold dağılımı:
  Fold 1: val=11200, val_target_ort=5.914
  Fold 2: val=11200, val_target_ort=5.914
  Fold 3: val=11200, val_target_ort=5.915
  Fold 4: val=11200, val_target_ort=5.909
  Fold 5: val=11200, val_target_ort=5.913


In [21]:
# HÜCRE 3.5 — Fold analizi
fold4_indices = None
for fold, (tr_i, va_i) in enumerate(kf_model.split(X, y_bins)):
    if fold == 3:  # Fold 4 (0-index)
        fold4_indices = va_i
        break

# Fold 4'teki veriyi inceleyelim
fold4_data = train_fe.iloc[fold4_indices]
fold4_target = y.iloc[fold4_indices]

print("=== FOLD 4 ANALİZİ ===")
print(f"Fold 4 target ort: {fold4_target.mean():.3f}")
print(f"Fold 4 target std: {fold4_target.std():.3f}")
print(f"Fold 4 target skew: {fold4_target.skew():.3f}")

# Diğer fold'larla karşılaştır
other_indices = [i for i in range(len(y)) if i not in fold4_indices]
other_target = y.iloc[other_indices]
print(f"\nDiğer fold'lar ort: {other_target.mean():.3f}")
print(f"Diğer fold'lar std: {other_target.std():.3f}")

# Feature dağılım farkı
print("\n=== Feature dağılım farkı (Fold 4 vs Diğerleri) ===")
for col in ['stres_skoru', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'gunluk_calisma_saati']:
    f4_mean = fold4_data[col].mean()
    other_mean = train_fe.loc[other_indices, col].mean()
    diff_pct = (f4_mean - other_mean) / other_mean * 100
    print(f"{col}: Fold4={f4_mean:.2f} vs Diğer={other_mean:.2f} (fark=%{diff_pct:+.1f})")

=== FOLD 4 ANALİZİ ===
Fold 4 target ort: 5.909
Fold 4 target std: 2.231
Fold 4 target skew: -0.291

Diğer fold'lar ort: 5.914
Diğer fold'lar std: 2.232

=== Feature dağılım farkı (Fold 4 vs Diğerleri) ===
stres_skoru: Fold4=5.72 vs Diğer=5.75 (fark=%-0.5)
rem_yuzdesi: Fold4=20.22 vs Diğer=20.24 (fark=%-0.1)
derin_uyku_yuzdesi: Fold4=20.23 vs Diğer=20.25 (fark=%-0.1)
gunluk_calisma_saati: Fold4=7.15 vs Diğer=7.16 (fark=%-0.2)


In [22]:
# HÜCRE 7 — Model Parametreleri

lgb_params = {
    'objective'        : 'regression',
    #'tweedie_variance_power' : 1.5,
    'boosting_type'          : 'gbdt',
    'metric'                 : 'rmse',
    'verbosity'              : -1,
    'random_state'           : SEED,
    'bagging_freq'           : 5,
    'learning_rate'          : 0.01,
    'num_leaves'             : 63,
    'min_child_samples'      : 50,
    'feature_fraction'       : 0.70,
    'bagging_fraction'       : 0.80,
    'reg_alpha'              : 0.2,
    'reg_lambda'             : 1.0,
    'min_split_gain'         : 0.01,
    'path_smooth'            : 0.1,
}

cat_params = dict(
    loss_function         = 'RMSE',
    eval_metric           = 'RMSE',
    iterations            = 6000,
    learning_rate         = 0.009,    # 0.008 ile 0.01 arası altın denge
    depth                 = 7,        # 7 ideal, 8 fazla gelmişti
    l2_leaf_reg           = 6.0,      # 10 çok fazlaydı, 5 biraz gevşekti. 6 ideal.
    random_strength       = 1.1,
    subsample             = 0.82,
    bootstrap_type        = 'Bernoulli',
    min_data_in_leaf      = 40,
    early_stopping_rounds = 400,
    random_seed           = SEED,
    verbose               = False,
)

xgb_params = {
    'objective'            : 'reg:squarederror',
    'eval_metric'          : 'rmse',
    'learning_rate'        : 0.01,
    'max_depth'            : 5,
    'min_child_weight'     : 100,
    'subsample'            : 0.70,
    'colsample_bytree'     : 0.60,
    'reg_alpha'            : 0.5,
    'reg_lambda'           : 3.0,
    'seed'                 : SEED,
    'verbosity'            : 0,
    'n_estimators'         : 5000,
    'early_stopping_rounds': 300,
}

print('Parametreler hazır.')

Parametreler hazır.


In [23]:
# HÜCRE 8 — 5-Fold CV Eğitimi

lgb_oof  = np.zeros(len(X))
cat_oof  = np.zeros(len(X))
xgb_oof  = np.zeros(len(X))
lgb_test = np.zeros(len(Xte))
cat_test = np.zeros(len(Xct))
xgb_test = np.zeros(len(Xte))
lgb_sc, cat_sc, xgb_sc = [], [], []

print('=== 5-Fold Stratified Ensemble ===\n')

for fold, (tr_i, va_i) in enumerate(kf_model.split(X, y_bins), 1):
    Xtr, Xva = X.iloc[tr_i], X.iloc[va_i]
    ytr, yva = y.iloc[tr_i], y.iloc[va_i]

    # CatBoost için Pool nesneleri — kategorik sütunlar BURADA bildirilir
    Xctr, Xcva = Xc.iloc[tr_i].copy(), Xc.iloc[va_i].copy()
    Xct_pred   = Xct.copy()

    # Kategorik sütun adlarını tespit et (string dtype olanlar)
    cat_col_names = [c for c in Xctr.columns if Xctr[c].dtype == object]

    pool_tr  = Pool(Xctr, label=ytr,  cat_features=cat_col_names)
    pool_va  = Pool(Xcva, label=yva,  cat_features=cat_col_names)
    pool_te  = Pool(Xct_pred,         cat_features=cat_col_names)

    print(f'--- Fold {fold} | val_ort={yva.mean():.3f} ---')

    # ── LightGBM ─────────────────────────────────────────────────────────────
    dt = lgb.Dataset(Xtr, label=ytr)
    dv = lgb.Dataset(Xva, label=yva, reference=dt)
    lm = lgb.train(
        lgb_params, dt, num_boost_round=5000,
        valid_sets=[dv],
        callbacks=[lgb.early_stopping(300, verbose=False),
                   lgb.log_evaluation(-1)]
    )
    lgb_oof[va_i] = lm.predict(Xva, num_iteration=lm.best_iteration)
    lgb_test     += lm.predict(Xte, num_iteration=lm.best_iteration) / N_FOLDS
    lgb_sc.append(np.sqrt(mean_squared_error(yva, lgb_oof[va_i])))
    print(f'  LGB  RMSE: {lgb_sc[-1]:.5f}  (iter={lm.best_iteration})')

    # ── CatBoost (Pool ile native kategorik) ──────────────────────────────────
    cm = CatBoostRegressor(**cat_params)
    cm.fit(pool_tr, eval_set=pool_va, use_best_model=True)
    cat_oof[va_i] = cm.predict(pool_va)
    cat_test     += cm.predict(pool_te) / N_FOLDS
    cat_sc.append(np.sqrt(mean_squared_error(yva, cat_oof[va_i])))
    print(f'  CAT  RMSE: {cat_sc[-1]:.5f}')

    # ── XGBoost ──────────────────────────────────────────────────────────────
    xm = xgb.XGBRegressor(**xgb_params)
    xm.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    xgb_oof[va_i] = xm.predict(Xva)
    xgb_test     += xm.predict(Xte) / N_FOLDS
    xgb_sc.append(np.sqrt(mean_squared_error(yva, xgb_oof[va_i])))
    print(f'  XGB  RMSE: {xgb_sc[-1]:.5f}\n')

lgb_r = np.sqrt(mean_squared_error(y, lgb_oof))
cat_r = np.sqrt(mean_squared_error(y, cat_oof))
xgb_r = np.sqrt(mean_squared_error(y, xgb_oof))

print('=== CV Özeti ===')
print(f'LGB OOF: {lgb_r:.5f}  (std=±{np.std(lgb_sc):.5f})')
print(f'CAT OOF: {cat_r:.5f}  (std=±{np.std(cat_sc):.5f})')
print(f'XGB OOF: {xgb_r:.5f}  (std=±{np.std(xgb_sc):.5f})')

=== 5-Fold Stratified Ensemble ===

--- Fold 1 | val_ort=5.914 ---
  LGB  RMSE: 1.21726  (iter=970)
  CAT  RMSE: 1.21092
  XGB  RMSE: 1.21483

--- Fold 2 | val_ort=5.914 ---
  LGB  RMSE: 1.22430  (iter=806)
  CAT  RMSE: 1.21710
  XGB  RMSE: 1.22143

--- Fold 3 | val_ort=5.915 ---
  LGB  RMSE: 1.21815  (iter=941)
  CAT  RMSE: 1.21389
  XGB  RMSE: 1.21797

--- Fold 4 | val_ort=5.909 ---
  LGB  RMSE: 1.23277  (iter=747)
  CAT  RMSE: 1.22536
  XGB  RMSE: 1.23089

--- Fold 5 | val_ort=5.913 ---
  LGB  RMSE: 1.21176  (iter=787)
  CAT  RMSE: 1.20712
  XGB  RMSE: 1.21123

=== CV Özeti ===
LGB OOF: 1.22087  (std=±0.00717)
CAT OOF: 1.21489  (std=±0.00619)
XGB OOF: 1.21929  (std=±0.00672)


In [24]:
df_err = train_c.copy()   # ham kategoriler burada duruyor
df_err['cat_pred'] = cat_oof
df_err['cat_err'] = abs(y.values - cat_oof)

print("=== gun_tipi ===")
print(df_err.groupby('gun_tipi')['cat_err'].agg(['mean', 'count']).sort_values('mean', ascending=False))

print("=== meslek x gun_tipi ===")
print(
    df_err.groupby(['meslek', 'gun_tipi'])['cat_err']
          .agg(['mean', 'count'])
          .query('count >= 100')
          .sort_values('mean', ascending=False)
          .head(20)
)

print("=== ruh_sagligi x gun_tipi ===")
print(
    df_err.groupby(['ruh_sagligi_durumu', 'gun_tipi'])['cat_err']
          .agg(['mean', 'count'])
          .query('count >= 100')
          .sort_values('mean', ascending=False)
)

=== gun_tipi ===
              mean  count
gun_tipi                 
0         1.009379  40052
1         0.869140  15948
=== meslek x gun_tipi ===
                                          mean  count
meslek                      gun_tipi                 
bilinmiyor                  0         1.095535    952
saglik personeli            0         1.070706   7040
muhendis                    0         1.059346   4649
satis ve pazarlama calisani 0         1.048399   2780
yonetici                    0         1.042938   3206
lojistik calisani           0         1.039455   2741
ogrenci                     0         1.038453   5885
avukat                      1         1.019711    779
                            0         1.017494   1963
egitimci                    0         1.007001   3122
lojistik calisani           1         0.981098   1070
saglik personeli            1         0.980097   2802
bilinmiyor                  1         0.934396    426
ev hanimi                   0         0.912

In [25]:
best_rmse = 999
best_w = None

for w_cat in np.arange(0.60, 0.96, 0.025):
    for w_xgb in np.arange(0.00, 0.31, 0.025):
        w_lgb = 1 - w_cat - w_xgb
        if w_lgb < 0:
            continue

        pred = w_cat * cat_oof + w_xgb * xgb_oof + w_lgb * lgb_oof
        rmse = np.sqrt(mean_squared_error(y, pred))

        if rmse < best_rmse:
            best_rmse = rmse
            best_w = (w_cat, w_xgb, w_lgb)

print(best_rmse, best_w)

1.2147794841237662 (np.float64(0.8750000000000002), np.float64(0.05), np.float64(0.07499999999999978))


In [26]:
pred_corr = pd.DataFrame({
    'lgb': lgb_oof,
    'cat': cat_oof,
    'xgb': xgb_oof
}).corr()

print(pred_corr)

err_corr = pd.DataFrame({
    'lgb_err': y.values - lgb_oof,
    'cat_err': y.values - cat_oof,
    'xgb_err': y.values - xgb_oof
}).corr()

print(err_corr)

          lgb       cat       xgb
lgb  1.000000  0.997278  0.998591
cat  0.997278  1.000000  0.997984
xgb  0.998591  0.997984  1.000000
          lgb_err   cat_err   xgb_err
lgb_err  1.000000  0.993597  0.996713
cat_err  0.993597  1.000000  0.995256
xgb_err  0.996713  0.995256  1.000000


In [27]:
# HÜCRE 9 — Ensemble + Ridge Meta
'''
# XGB ağırlığı %5 ile sınırlı
inv_lgb = 1.0 / (lgb_r + 1e-12)
inv_cat = 1.0 / (cat_r + 1e-12)
w_xgb   = 0.05
rem     = 1.0 - w_xgb
w_lgb   = rem * inv_lgb / (inv_lgb + inv_cat)
w_cat   = rem * inv_cat / (inv_lgb + inv_cat)

tree_oof  = w_lgb * lgb_oof  + w_cat * cat_oof  + w_xgb * xgb_oof
tree_test = w_lgb * lgb_test + w_cat * cat_test + w_xgb * xgb_test

tree_rmse = np.sqrt(mean_squared_error(y, tree_oof))
print(f'Ağaç Ensemble OOF: {tree_rmse:.5f}')
print(f'Ağırlıklar → LGB:{w_lgb:.3f}  CAT:{w_cat:.3f}  XGB:{w_xgb:.3f}')

# Ridge meta-learner (StratifiedKFold ile — sızıntı yok)
oof_stack  = np.column_stack([lgb_oof, cat_oof, xgb_oof, tree_oof])
test_stack = np.column_stack([lgb_test, cat_test, xgb_test, tree_test])

scaler    = StandardScaler()
oof_sc    = scaler.fit_transform(oof_stack)
test_sc   = scaler.transform(test_stack)

ridge_oof  = np.zeros(len(y))
ridge_test = np.zeros(len(Xte))

for tr_i, va_i in kf_model.split(oof_sc, y_bins):
    rr = Ridge(alpha=50.0)
    rr.fit(oof_sc[tr_i], y.iloc[tr_i])
    ridge_oof[va_i] = rr.predict(oof_sc[va_i])
    ridge_test     += rr.predict(test_sc) / N_FOLDS

ridge_rmse = np.sqrt(mean_squared_error(y, ridge_oof))
print(f'Ridge Meta OOF: {ridge_rmse:.5f}')

RIDGE_W    = 0.10
final_oof  = (1 - RIDGE_W) * tree_oof  + RIDGE_W * ridge_oof
final_test = (1 - RIDGE_W) * tree_test + RIDGE_W * ridge_test
final_rmse = np.sqrt(mean_squared_error(y, final_oof))
print(f'Final Blend OOF: {final_rmse:.5f}')

print('\n=== Fold bazlı RMSE (sapma kontrolü) ===')
fold_rmses = []
for fi, (_, va_i) in enumerate(kf_model.split(X, y_bins), 1):
    fr = np.sqrt(mean_squared_error(y.iloc[va_i], final_oof[va_i]))
    fold_rmses.append(fr)
    flag = ' ← sapma!' if fr > np.mean(fold_rmses) + 0.010 else ''
    print(f'  Fold {fi}: {fr:.5f}{flag}')
print(f'  Std: ±{np.std(fold_rmses):.5f}')  '''

"\n# XGB ağırlığı %5 ile sınırlı\ninv_lgb = 1.0 / (lgb_r + 1e-12)\ninv_cat = 1.0 / (cat_r + 1e-12)\nw_xgb   = 0.05\nrem     = 1.0 - w_xgb\nw_lgb   = rem * inv_lgb / (inv_lgb + inv_cat)\nw_cat   = rem * inv_cat / (inv_lgb + inv_cat)\n\ntree_oof  = w_lgb * lgb_oof  + w_cat * cat_oof  + w_xgb * xgb_oof\ntree_test = w_lgb * lgb_test + w_cat * cat_test + w_xgb * xgb_test\n\ntree_rmse = np.sqrt(mean_squared_error(y, tree_oof))\nprint(f'Ağaç Ensemble OOF: {tree_rmse:.5f}')\nprint(f'Ağırlıklar → LGB:{w_lgb:.3f}  CAT:{w_cat:.3f}  XGB:{w_xgb:.3f}')\n\n# Ridge meta-learner (StratifiedKFold ile — sızıntı yok)\noof_stack  = np.column_stack([lgb_oof, cat_oof, xgb_oof, tree_oof])\ntest_stack = np.column_stack([lgb_test, cat_test, xgb_test, tree_test])\n\nscaler    = StandardScaler()\noof_sc    = scaler.fit_transform(oof_stack)\ntest_sc   = scaler.transform(test_stack)\n\nridge_oof  = np.zeros(len(y))\nridge_test = np.zeros(len(Xte))\n\nfor tr_i, va_i in kf_model.split(oof_sc, y_bins):\n    rr = Ridge

In [28]:
# HÜCRE 10 — Şampiyonluk Karışımı
# En iyi model CatBoost, sonra XGB, en son dengeleyici olarak LGB
# Toplamı 1.0 (100%) olmalı.
final_preds = (cat_test * 0.65) + (xgb_test * 0.25) + (lgb_test * 0.10)

#test_preds = final_preds

#submission = pd.DataFrame({
#    'id'                       : test_id,
 #3})

#submission.to_csv('final_master_blend.csv', index=False, float_format='%.6f')
#print("Stratejik Blend tamam: Cat(65) + XGB(25) + LGB(10)")

In [29]:
base_blend = 0.875 * cat_oof + 0.125 * lgb_oof
print(np.sqrt(mean_squared_error(y, base_blend)))

1.2147847429981256


## Troubleshooting — Sonuçlar Beklenenden Kötüyse

### A: CatBoost hata verirse
```python
# Kategorik sütunları kontrol et
print(Xc.dtypes)  # cat_col_names'teki sütunlar 'object' dtype olmalı
# NaN flag sütunları int olmalı
print(train_nan.dtypes)
```

### B: XGB hala unstable ise — tamamen kaldır
```python
w_lgb = inv_lgb / (inv_lgb + inv_cat)
w_cat = inv_cat / (inv_lgb + inv_cat)
tree_oof  = w_lgb * lgb_oof  + w_cat * cat_oof
tree_test = w_lgb * lgb_test + w_cat * cat_test
```

### C: Fold 5 hala sapıyorsa
```python
SEED = 123  # veya 0, 7, 99
# ya da N_BINS = 20
```

### D: Optuna (vaktin varsa, ~15-30dk)
```python
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def cat_objective(trial):
    params = dict(
        loss_function='RMSE', iterations=2000, verbose=False,
        random_seed=SEED, early_stopping_rounds=100,
        learning_rate = trial.suggest_float('lr', 0.005, 0.03, log=True),
        depth         = trial.suggest_int('depth', 5, 9),
        l2_leaf_reg   = trial.suggest_float('l2', 3.0, 15.0),
        random_strength = trial.suggest_float('rs', 0.5, 3.0),
        min_data_in_leaf = trial.suggest_int('min_leaf', 20, 100),
    )
    scores = []
    for tr_i, va_i in StratifiedKFold(3, shuffle=True, random_state=SEED).split(Xc, y_bins):
        p_tr = Pool(Xc.iloc[tr_i], label=y.iloc[tr_i],
                    cat_features=[c for c in Xc.columns if Xc[c].dtype==object])
        p_va = Pool(Xc.iloc[va_i], label=y.iloc[va_i],
                    cat_features=[c for c in Xc.columns if Xc[c].dtype==object])
        m = CatBoostRegressor(**params)
        m.fit(p_tr, eval_set=p_va, use_best_model=True)
        scores.append(np.sqrt(mean_squared_error(y.iloc[va_i], m.predict(p_va))))
    return np.mean(scores)

study = optuna.create_study(direction='minimize')
study.optimize(cat_objective, n_trials=50, show_progress_bar=True)
print(f'En iyi CAT RMSE: {study.best_value:.5f}')
print(study.best_params)
```